# Semantic Mapping of Mental Health Survey Questions

## 1. Overview

This notebook implements **Method 1 (Baseline): Semantic Mapping** for the mental health dimension reduction project.

The goal of this method is to map diverse mental health survey questions onto a shared set of **high-level wellness dimensions** (Emotional, Social, Physical, Occupational, Intellectual, Spiritual, Environmental, Financial) using a **concept-driven, label-free approach**.

Rather than learning dimension representations from labeled data, this approach begins by defining the *same set of wellness dimension types* using **natural-language definitions generated by multiple large language models (LLMs)**.  
Each LLM is prompted with an identical instruction, producing alternative textual realizations of the same conceptual dimensions.

These model-generated dimension definitions serve as **semantic prototypes (anchors)**. Both survey questions and dimension prototypes are embedded into a shared semantic space using a pretrained sentence encoder. Survey questions are then mapped back to the wellness dimensions by measuring cosine similarity to the corresponding prototypes, allowing each question to be assigned to one or more dimensions.

By holding the dimension types fixed while varying the language used to define them, this method enables controlled comparisons of how different LLM-generated definitions influence semantic alignment and downstream mappings. Semantic mapping thus provides a principled bridge between conceptual wellness frameworks and data-driven analysis.

## 2. Data

Each data point corresponds to a **single survey question**, represented with:
- `qid`: the original item identifier (e.g., PSQI_5_3, PSS_12)
- `dataset`: the source questionnaire or scale
- `text`: the question text

All questions are consolidated into a canonical dataset (`questions_master`) during preprocessing to ensure:
- No modification of raw source files
- Consistent identifiers across methods
- Reproducibility across models and experiments


In [1]:
from pathlib import Path
from mhdr.dataloader.io import read_csv, save_csv
import pandas as pd
INPUT_DIR = Path.cwd() / "input"
OUTPUT_DIR = Path.cwd() / "output"
TEMP_DIR = Path.cwd() / "temp"

questions = read_csv(INPUT_DIR / "questions.csv")
questions.head()


,qid,text,dataset
0,CD_RISC_1,I am able to adapt when changes occur.,CD-RISC
1,CD_RISC_2,I have one close and secure relationship.,CD-RISC
2,CD_RISC_3,Sometimes fate or God helps me.,CD-RISC
3,CD_RISC_4,I can deal with whatever comes my way.,CD-RISC
4,CD_RISC_5,Past successes give me confidence.,CD-RISC


In [2]:
print("Total questions:", len(questions))
questions["dataset"].value_counts()

Total questions: 145


dataset
PWS        36
CD-RISC    25
PERMA      23
PSS        23
UCLA       20
PWB        18
Name: count, dtype: int64

## 3. Dimension
The wellness dimensions used in this project are represented by natural-language definitions generated by multiple large language models (LLMs).
Each model is prompted using the same fixed instruction, ensuring that differences across dimension sets reflect model-specific generation behavior rather than prompt variation.


- `model_name`: the identifier of the large language model used to generate the definition  
- `dim_name`: the name of the wellness dimension (e.g., Emotional, Social, Physical)  
- `dim_text`: the natural-language definition generated for that dimension
A utility function (`load_dimension_sets`) is provided to load this file and group the definitions by model.  
The resulting data structure is a dictionary mapping each model name to a list of its corresponding dimension definitions, for example:

```python
{
  "ChatGPT-5.2": [
    "Emotional: ...",
    "Environmental: ...",
    "Financial: ...",
    "Intellectual: ...",
    "Occupational: ...",
    "Physical: ...",
    "Social: ...",
    "Spiritual: ..."
  ],
  "DeepSeek-V3.2": [...],
  "Llama-4": [...],
  "claude-sonnet-4.5": [...],
  "gemini-3.0-pro": [...]
}
```
Each model-specific dimension set is used independently as a collection of semantic prototypes.
These prototypes are mapped against the survey questions described in Section 2 (Data), enabling controlled comparisons of how different LLM-generated dimension definitions influence downstream question-to-dimension mappings.

In [3]:
from mhdr.model.LLMio import load_dimension_sets
dimension_sets = load_dimension_sets(INPUT_DIR / "dim_definations.csv")
print(dimension_sets)
print(len(dimension_sets), len(dimension_sets["Llama-4"]))

{'ChatGPT-5.2': ['Emotional: The ability to recognize, understand, express, and manage feelings in ways that support resilience, balance, and healthy responses to life events.', 'Environmental: The quality of interaction with surrounding spaces, resources, and conditions that influence comfort, safety, and daily functioning.', 'Financial: The capacity to manage money decisions and resources in ways that support stability, choice, and future needs.', 'Intellectual: The ongoing engagement with learning, curiosity, creativity, and critical thinking to expand understanding and skills.', 'Occupational: The sense of purpose, satisfaction, and alignment experienced through work or meaningful daily roles.', 'Physical: The state of the body and the habits that support energy, strength, mobility, and overall functioning.', 'Social: The ability to build, maintain, and experience supportive relationships and a sense of belonging.', 'Spiritual: The way meaning, values, and connection guide a person

## 4. Semantic Mapping Using LLM-Generated Dimensions

In this stage, we perform semantic mapping between survey questions and
wellness dimensions using the `SemanticMapper` module
(`model/mapping/semantic_mapper.py`).

### Mapping Procedure

For each set of dimension definitions generated by a large language model (LLM),
we independently perform a full semantic mapping process.
This results in **one mapping output per LLM**, allowing us to compare
consistency and variation across models.

The mapping procedure consists of the following steps:

1. **Encoding**
   - Survey questions and wellness dimension definitions are embedded into a
     shared semantic space using a pretrained sentence embedding model
     (`all-MiniLM-L6-v2`).
   - The same encoder is used for both questions and dimensions to ensure
     comparability.

2. **Similarity Computation**
   - For each question, cosine similarity is computed against all dimension
     embeddings, producing a question–dimension similarity matrix.

3. **Margin-Based Assignment**
   - Each question is first assigned to the dimension with the highest
     similarity score.
   - Additional dimensions are included if their similarity scores fall within
     a predefined margin (`delta`) of the top score.
   - This allows a single question to be assigned to **multiple dimensions**
     when semantic overlap exists, reflecting the multifaceted nature of
     mental health constructs.

## Output Structure

Each mapping run produces a table (CSV) where **each row corresponds to one survey question mapped under one LLM-generated dimension set**. Conceptually, each row has the following structure:

```python
{
  "qid": "CD_RISC_1",
  "dataset": "CD-RISC",
  "text": "I am able to adapt when changes occur.",
  "dimension_model": "ChatGPT-5.2",
  "dimensions": ["Emotional", "Spiritual", "Physical"],
  "scores": [0.18, 0.17, 0.16]
}
```
- **`qid`**  
  Unique identifier of the survey question.

- **`dataset`**  
  Source questionnaire (e.g., CD-RISC, PERMA, PSS, UCLA).

- **`text`**  
  Original survey question text.

- **`dimension_model`**  
  The LLM used to generate the wellness dimension definitions for this run.

- **`dimensions`**  
  One or more assigned wellness dimensions.  
  Multiple dimensions may be selected when their similarity scores fall within a margin (Δ) of the best match.

- **`scores`**  
  Cosine similarity scores corresponding to the listed dimensions (same order).


In [4]:
from mhdr.model.semantic_mapper import SemanticMapper

deltas = [0.05]

# 1) init once
mapper = SemanticMapper()
mapper.set_questions_df(questions)  # encode once

all_results = []

for model_name, dim_defs in dimension_sets.items():
    print(f"\n=== Mapping using {model_name} ===")
    mapper.set_dimensions(dim_defs, dimension_model_name=model_name)

    for delta in deltas:
        mapped = mapper.map_questions_to_dimensions(delta=delta)
        mapped = mapped.copy()
        mapped["delta"] = float(delta)

        save_csv(mapped, OUTPUT_DIR / f"mapped_{model_name}_delta{delta}.csv")

        all_results.append(mapped)

# 2) merge
all_mapped = pd.concat(all_results, ignore_index=True)

# 3) sanity checks
print("\nMerged shape:", all_mapped.shape)
print("models in merged:", all_mapped["dimension_model"].nunique())
print(all_mapped["dimension_model"].value_counts())
print("deltas in merged:", sorted(all_mapped["delta"].unique()))


/Users/haikeyu/Desktop/mentalhealth-dimension-reduction/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1735.88it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== Mapping using ChatGPT-5.2 ===

=== Mapping using DeepSeek-V3.2 ===

=== Mapping using Llama-4 ===

=== Mapping using claude-sonnet-4.5 ===

=== Mapping using gemini-3.0-pro ===

Merged shape: (725, 7)
models in merged: 5
dimension_model
ChatGPT-5.2          145
DeepSeek-V3.2        145
Llama-4              145
claude-sonnet-4.5    145
gemini-3.0-pro       145
Name: count, dtype: int64
deltas in merged: [np.float64(0.05)]
